<a target="_blank" href="https://colab.research.google.com/github/univiemops/tewa1-computational-cognition/blob/main/extra/07%20Recap.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


Recap lab 7
===

## Imports and prerequisites

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.linear_model import LinearRegression

response = requests.get(
    "https://ucloud.univie.ac.at/index.php/s/crRApfnS4HEm2ar/download"
)
open("real_estate.csv", "wb").write(response.content)

df = pd.read_csv("real_estate.csv")

y_name = "Y house price of unit area"
x_names = list(df.drop(["No", y_name], axis=1))

##  Exercise 1: Feature selection

Let's now focus on selecting the most important features or variables from our dataset to create our model. Our goal is to improve model performance by minimizing dimensionality and complexity. To achieve this, we will

 1. Add the predictors **sequentially**, i.e., first fit a model to only X1, then another model to X1 and X2, until you have included all the features up to X6, store the resulting `score()` (R²) for each of the models.
 2. Do the same, but add predictors in a **random** order one-by-one.
 3. Do the same, but add predictors in the order of the **pearson correlation** with the outcome variable Y (starting with the largest).
 4. Plot the obtained scores from 1-3 as three lines.
 
You can use for-loops to iterate over the different models.

Please add a sentence about what you are observing.

In [ ]:
corr_Xy = (
    df.drop("No", axis=1).corr()[[y_name]].sort_values(y_name).drop(y_name, axis=0)
)
corr_Xy_abs = corr_Xy.abs().sort_values(y_name, ascending=False)
corr_Xy_abs

In [ ]:
# define plot structure

fig, ax = plt.subplots(3, 1, figsize=(6, 12))
ylim_max = 0.7

# plot pre-defined order

scores_pre = []

for i, x_name in enumerate(x_names):

    lr = LinearRegression()
    lr.fit(df[x_names[: i + 1]], df[y_name])
    scores_pre.append(lr.score(df[x_names[: i + 1]], df[y_name]))

fig.suptitle("Predictor inclusion for " + str(lr)[:-2] + " model")

ax[0].plot(x_names, scores_pre, marker="o")
ax[0].set_xticks(ax[0].get_xticks())
ax[0].set_xticklabels([name[:12] + "..." for name in x_names], rotation=45, ha="right")
ax[0].set_ylabel("R²")
ax[0].set_title("in pre-defined order")
ax[0].set_ylim(0, ylim_max)
ax[0].grid()

# plot random order

np.random.seed(0)

x_names_random = np.random.choice(x_names, len(x_names), replace=False)

scores_random = []

for i, x_name in enumerate(x_names_random):

    lr = LinearRegression()
    lr.fit(df[x_names_random[: i + 1]], df[y_name])
    scores_random.append(lr.score(df[x_names_random[: i + 1]], df[y_name]))

ax[1].plot(x_names_random, scores_random, marker="o")
ax[1].set_xticks(ax[0].get_xticks())
ax[1].set_xticklabels(
    [name[:12] + "..." for name in x_names_random], rotation=45, ha="right"
)
ax[1].set_ylabel("R²")
ax[1].set_title("in some random order")
ax[1].set_ylim(0, ylim_max)
ax[1].grid()

# plot order by correlation (highest first)

x_names_corr = list(corr_Xy_abs.T)

scores_corr = []

for i, x_name in enumerate(x_names_corr):

    lr = LinearRegression()
    lr.fit(df[x_names_corr[: i + 1]], df[y_name])
    scores_corr.append(lr.score(df[x_names_corr[: i + 1]], df[y_name]))

ax[2].plot(x_names_corr, scores_corr, marker="o")
ax[2].set_xticks(ax[0].get_xticks())
ax[2].set_xticklabels(
    [name[:12] + "..." for name in x_names_corr], rotation=45, ha="right"
)
ax[2].set_ylabel("R²")
ax[2].set_title("in order of correlation (highest first)")
ax[2].set_ylim(0, ylim_max)
ax[2].grid()

# make space for labels and avoid overlaps

fig.tight_layout()

Alternative Visualization Without Predictor Names

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(scores_pre, label="Pre-defined order", marker="o")
plt.plot(scores_random, label="Random order", marker="o")
plt.plot(scores_corr, label="In order of correlation", marker="o")
plt.xticks(range(len(x_names)), range(1, 1 + len(x_names)))
plt.title("Predictor inclusion for " + str(lr)[:-2] + " model")
plt.legend()
plt.grid()
plt.show()

*All methods end up with the same R2 value. If we want to keep our model simple and include only a few predictors, it makes sense to focus on the predictors that explain the most variance. This means adding predictors in descending order of their correlation with the outcome. There is also a good deal of multicollinearity between latitude and longitude. Although both are significantly correlated with outcome, they appear to be explaining the same "thing" and therefore one does not add value over the other. Multicollinearity is generally an important issue when including predictors in the model step by step, since the order of inclusion determines how much the remaining variables influence the variance of the outcome.*

## Exercise 2: Train-test split

**Overfitting** is a common problem in machine learning, where the model learns too well from the training data, even picking up noise and irrelevant details, which makes it difficult to generalize to new data. The train-test split helps by splitting the data into two parts: one for training the model and another for testing how well it generalizes to new data. There is a built-in train-test-split function in scikit-learn, but we want you to build your own. To achieve this, we will:

1. Split the `X` and `y` data into a 80% training (`X_train`, `y_train`) and a 20% test set (`X_test`, `y_test`).

- Option 1: Simply use the first 80% of rows for the training set and the rest for the test set). You can use indexing for this: e.g., `df[0:int(len(Data)*.8)]` selects the first 80% percent of a numpy array.

- Option 2: Randomly select 80% of the data as training, and the rest for test. This is the better approach, but make sure that both X and y are created from the same random selection (e.g., the rows in `X_test` correspond to the rows in `y_test`, ...).


2. Fit the model to the training set, and compute the `score()` both for the training and the test set

3. Similar to the previous exercise, try to find the best combination of predictors that best explain the test set and compute the `score()` for both training set and test set.

4. Try to visualize the last results in a similar way like in Exercise 1 (one plot with a line for training set and a line for the test set).

Please add a sentence about what you are observing.

In [ ]:
# 1 Split data into training and testing

X = df[x_names]
y = df[y_name]

np.random.seed(0)

train_idx = sorted(np.random.choice(range(len(df)), round(len(df) * 0.8)))
test_idx = np.array(range(len(df)))[
    ~np.array([item in train_idx for item in range(len(df))])
]

X_train = X.iloc[train_idx, :]
y_train = y[train_idx]
X_test = X.iloc[test_idx, :]
y_test = y[test_idx]

# 2 Fit the model and calculate R2

lr.fit(X_train, y_train)
print("R2 for train set: ", lr.score(X_train, y_train))
print("R2 for test set:  ", lr.score(X_test, y_test))

In [ ]:
#  3. find combination of predictors that best explain the test set and computing R2 for train and test set

x_names_corr = list(corr_Xy_abs.T)

scores_train = []
scores_test = []

for i, x_name in enumerate(x_names_corr):
    lr = LinearRegression()
    lr.fit(X_train[x_names_corr[: i + 1]], y_train)
    scores_train.append(lr.score(X_train[x_names_corr[: i + 1]], y_train))
    scores_test.append(lr.score(X_test[x_names_corr[: i + 1]], y_test))

# Plotting R^2 for different variable combinations of test set
plt.figure(figsize=(7, 5))
plt.title("Predictor inclusion in order of correlation")
plt.plot(scores_train, label="Train", marker="o")
plt.plot(scores_test, label="Test", marker="o")
plt.ylabel("R2")
plt.ylim(0, 1)
plt.xticks(range(len(x_names)), range(1, 1 + len(x_names)))
plt.legend()
plt.grid()
plt.show()

*R2 is consistently higher for the test data. This can happen if we have only one (random) split and leads to an overestimation of the true generalizability of our model. If we shuffle the data differently (e.g., using random seed of 3 instead of 0) and then get different splits, we will get slightly different R2 values. If we do this several times and average the resulting R2 values for the test set, we will get a more robust estimate of the model's ability to generalize to new data. This procedure is called **cross validation**.*